<a href="https://colab.research.google.com/github/MayerT1/LiDAR_Dev/blob/main/Sewanee_Field_Protocol_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# !pip install geopandas geemap ee
!pip install geemap geopandas scikit-learn tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 23.4 MB/s eta 0:00:00


In [2]:
# !pip install geopandas geemap ee

import geopandas as gpd
import pandas as pd
import ee
import geemap
from shapely.geometry import Point
import os


In [4]:
from google.colab import drive
drive.mount('/content/drive/')

ValueError: mount failed

In [ ]:
ee.Authenticate()
ee.Initialize(project='servir-sco-assets')
Map = geemap.Map()

In [ ]:
# Load AOI from GEE
aoi = ee.FeatureCollection('projects/servir-sco-assets/assets/Rx_Fire/Vector_Data/Sewanee_Domain').geometry().buffer(100)
aoi = ee.FeatureCollection(aoi)

# Convert AOI to GeoPandas using geemap
aoi_gdf = geemap.ee_to_gdf(aoi)


In [ ]:
import glob

# Path to GEDI CSVs
csv_folder = '/content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/target_data/filtered_csvs'
csv_files = glob.glob(os.path.join(csv_folder, '*.csv'))

# Load and concatenate all CSVs
dfs = [pd.read_csv(f) for f in csv_files]
gedi_df = pd.concat(dfs, ignore_index=True)

# Convert to GeoDataFrame
geometry = [Point(xy) for xy in zip(gedi_df['lon'], gedi_df['lat'])]
gedi_gdf = gpd.GeoDataFrame(gedi_df, geometry=geometry, crs='EPSG:4326')


In [ ]:
# Ensure same CRS
gedi_gdf = gedi_gdf.to_crs(aoi_gdf.crs)

# Spatial join: GEDI points inside AOI
gedi_filtered = gpd.sjoin(gedi_gdf, aoi_gdf, how="inner", predicate='intersects')


In [ ]:
print(f"Total GEDI points: {len(gedi_gdf)}")
print(f"Filtered GEDI points: {len(gedi_filtered)}")


In [ ]:
split_creek = ee.FeatureCollection('projects/servir-sco-assets/assets/Rx_Fire/Vector_Data/Split_Creek_Observatory')

# Convert trees to GeoPandas using geemap
split_creek_gdf = geemap.ee_to_gdf(split_creek)


In [ ]:
trees = ee.FeatureCollection('projects/servir-sco-assets/assets/Rx_Fire/Vector_Data/Split_Creek_Ob')

# Convert trees to GeoPandas using geemap
trees_gdf = geemap.ee_to_gdf(trees)


In [ ]:
comp_12 = ee.FeatureCollection('projects/servir-sco-assets/assets/Rx_Fire/Vector_Data/Comp_12')

# Convert trees to GeoPandas using geemap
comp_12_gdf = geemap.ee_to_gdf(comp_12)


In [ ]:
mgmt = ee.FeatureCollection('projects/servir-sco-assets/assets/Rx_Fire/Vector_Data/Sewanee_MGMT')

# Convert trees to GeoPandas using geemap
mgmt_gdf = geemap.ee_to_gdf(mgmt)


In [ ]:
# 1. Convert to projected CRS for accurate buffering
utm_crs = 'EPSG:32616'  # UTM zone (adjust if needed)
gedi_proj = gedi_filtered.to_crs(utm_crs)

# 2. Buffer to 12.5m radius (25m diameter)
buffer_radius = 12.5
gedi_buffers = gedi_proj.copy()
gedi_buffers['geometry'] = gedi_proj.geometry.buffer(buffer_radius)

# 3. Remove overlapping buffers (keep one per overlapping group)
# Build spatial index
sindex = gedi_buffers.sindex

# Track IDs of buffers to keep
keep_idxs = set()
visited = set()

for idx, geom in gedi_buffers.geometry.items():
    if idx in visited:
        continue
    # Find all others that intersect this one
    possible_matches_index = list(sindex.intersection(geom.bounds))
    overlapping = gedi_buffers.iloc[possible_matches_index]
    overlapping = overlapping[overlapping.geometry.intersects(geom)]

    # Keep only the first from the overlapping group
    to_keep = overlapping.index[0]
    keep_idxs.add(to_keep)

    # Mark all overlapping as visited
    visited.update(overlapping.index)

# Keep only non-overlapping plots
gedi_buffers = gedi_buffers.loc[list(keep_idxs)].copy()

# 4. Convert back to EPSG:4326 for web mapping
gedi_buffers = gedi_buffers.to_crs('EPSG:4326')

# 5. Add to map
Map.add_gdf(gedi_buffers, layer_name='Unique Non-Overlapping GEDI 25m Plots')
Map


In [ ]:
# # 6. Convert AOIs to GeoDataFrames in same CRS
# domain_gdf = geemap.ee_to_gdf(aoi).to_crs('EPSG:4326')
# mgmt_gdf = geemap.ee_to_gdf(mgmt).to_crs('EPSG:4326')
# split_gdf = geemap.ee_to_gdf(split_creek).to_crs('EPSG:4326')
# comp12_gdf = geemap.ee_to_gdf(comp_12).to_crs('EPSG:4326')

# 7. Union of sub-AOIs
sub_aois_union = mgmt_gdf.unary_union.union(split_creek_gdf.unary_union).union(comp_12_gdf.unary_union)

# 8. Subtract sub-AOIs from domain
domain_minus_subs = aoi_gdf.overlay(gpd.GeoDataFrame(geometry=[sub_aois_union], crs='EPSG:4326'), how='difference')

# 9. Select plots within the domain minus sub-AOIs
gedi_domain_only = gpd.sjoin(gedi_buffers, domain_minus_subs, how='inner', predicate='intersects', rsuffix='domain')

# 10. Sample desired number of plots from domain-only
n = 25  # 👈 change this to how many you want
domain_sampled = gedi_domain_only.sample(n=min(n, len(gedi_domain_only)), random_state=42)

# 11. Add to map
Map.add_gdf(domain_sampled, layer_name=f'{n} Domain-Only Plots')
Map


remove extra plots for sub aois

In [ ]:
def sample_plots_within_aoi(plots_gdf, aoi_gdf, max_plots):
    # Spatial join to get plots within AOI
    plots_in_aoi = gpd.sjoin(plots_gdf, aoi_gdf, how='inner', predicate='intersects', rsuffix='AOI')

    # Randomly sample max_plots or fewer if less available
    sampled = plots_in_aoi.sample(n=min(len(plots_in_aoi), max_plots), random_state=42)

    return sampled


In [ ]:
max_split_creek = 20
max_comp_12 = 20
max_mgmt = 45
# max_domain = 15

split_creek_sampled = sample_plots_within_aoi(gedi_buffers, split_creek_gdf, max_split_creek)
comp_12_sampled = sample_plots_within_aoi(gedi_buffers, comp_12_gdf, max_comp_12)
mgmt_sampled = sample_plots_within_aoi(gedi_buffers, mgmt_gdf, max_mgmt)
# domain_sampled = sample_plots_within_aoi(gedi_buffers, aoi_gdf, max_domain)


merge a all plots across al aois

In [ ]:
import pandas as pd

sampled_plots = pd.concat([split_creek_sampled, comp_12_sampled, mgmt_sampled, domain_sampled]).drop_duplicates(subset=gedi_buffers.columns)
print(sampled_plots.head())

In [ ]:
# Create Map
Map = geemap.Map(center=[35.2, -85.9], zoom=13)

# Add comp 12
Map.add_gdf(comp_12_gdf, layer_name="comp 12")

# Add comp 12
Map.add_gdf(mgmt_gdf, layer_name="mgmt")

# Add comp 12
Map.add_gdf(split_creek_gdf, layer_name="Split Creek Observatory AOI")

# Add AOI
Map.add_gdf(aoi_gdf, layer_name="Domain")

# add trees
# Map.add_gdf(trees_gdf, layer_name="Split Creek Ob trees")

# Add GEDI points (filtered)
Map.add_gdf(gedi_filtered, layer_name="Filtered GEDI Points")

# Step 8: Add to geemap
# Add to map
Map.add_gdf(sampled_plots, layer_name='sampled_plots')

Map


In [ ]:
import random
from shapely.geometry import Point
from shapely.ops import unary_union

def generate_random_plots(aoi_gdf, existing_buffers, n, radius_m=12.5, crs='EPSG:32616'):
    """
    Generate n non-overlapping circular plots of given radius inside an AOI,
    avoiding overlaps with existing buffers.
    """
    # Project to a CRS in meters for buffering & random point generation
    aoi_proj = aoi_gdf.to_crs(crs)
    existing_proj = existing_buffers.to_crs(crs)

    # Merge existing buffers into one geometry to check overlaps
    existing_union = unary_union(existing_proj.geometry)

    plots = []
    bounds = aoi_proj.total_bounds  # xmin, ymin, xmax, ymax

    attempts = 0
    max_attempts = n * 500  # safety limit

    while len(plots) < n and attempts < max_attempts:
        attempts += 1
        # Generate a random point within AOI bounding box
        x = random.uniform(bounds[0], bounds[2])
        y = random.uniform(bounds[1], bounds[3])
        pt = Point(x, y)

        # Check if inside AOI polygon
        if not aoi_proj.unary_union.contains(pt):
            continue

        # Create circular buffer
        circle = pt.buffer(radius_m)

        # Skip if overlaps existing buffers or already accepted plots
        if circle.intersects(existing_union) or any(circle.intersects(p) for p in plots):
            continue

        plots.append(circle)

    # Build GeoDataFrame
    plots_gdf = gpd.GeoDataFrame(geometry=plots, crs=crs).to_crs('EPSG:4326')
    return plots_gdf

# Generate 10 plots for each AOI
mgmt_new = generate_random_plots(mgmt_gdf, gedi_buffers, 10)
comp12_new = generate_random_plots(comp_12_gdf, gedi_buffers, 10)
split_new = generate_random_plots(split_creek_gdf, gedi_buffers, 10)
aoi_new = generate_random_plots(aoi_gdf, gedi_buffers, 10)

non_GEDI_Plots = pd.concat([aoi_new, split_new, comp12_new, mgmt_new])#.drop_duplicates(subset=gedi_buffers.columns)


# Add to map
# Map.add_gdf(mgmt_new, "10 New MGMT Plots")
# Map.add_gdf(comp12_new, "10 New Comp_12 Plots")
# Map.add_gdf(split_new, "10 New Split_Creek Plots")
Map.add_gdf(non_GEDI_Plots, "non_GEDI_Plots")
Map


Plots with no GEDI centorid

In [ ]:
# from shapely.ops import unary_union

# # 1. Ensure everything is in the same CRS for spatial checks
# gedi_proj = gedi_buffers.to_crs('EPSG:32616')
# sampled_proj = sampled_plots.to_crs('EPSG:32616')

# # 2. Merge existing plots into a single geometry
# existing_union = unary_union(sampled_proj.geometry)

# # 3. Remove any gedi_buffers that overlap existing plots
# gedi_non_overlap = gedi_proj[~gedi_proj.geometry.intersects(existing_union)]

# # 4. Randomly select 25 plots from the remaining ones
# n = 25
# domain_sampled = gedi_non_overlap.sample(
#     n=min(n, len(gedi_non_overlap)),
#     random_state=42
# )

# # 5. Convert back to EPSG:4326 for mapping
# domain_sampled = domain_sampled.to_crs('EPSG:4326')

# # 6. Add to map
# Map.add_gdf(domain_sampled, layer_name=f'{n} Fix Non-Overlapping Plots from GEDI Buffers')
# Map


In [ ]:
# import random
# from shapely.geometry import Point
# from shapely.ops import unary_union
# import geopandas as gpd

# # 1. Convert AOI to GeoDataFrame (if not already)
# domain_gdf = geemap.ee_to_gdf(aoi).to_crs('EPSG:32616')

# # 2. Project sampled_plots to match (needed for accurate distance-based checks)
# sampled_proj = sampled_plots.to_crs('EPSG:32616')
# sampled_union = unary_union(sampled_proj.geometry)

# # 3. Prepare to generate random points
# minx, miny, maxx, maxy = domain_gdf.total_bounds
# domain_polygon = unary_union(domain_gdf.geometry)

# buffer_radius = 12.5
# new_plots = []
# attempts = 0
# max_attempts = 10000  # fail-safe to prevent infinite loops

# while len(new_plots) < 75 and attempts < max_attempts:
#     x = random.uniform(minx, maxx)
#     y = random.uniform(miny, maxy)
#     point = Point(x, y)
#     if not domain_polygon.contains(point):
#         attempts += 1
#         continue

#     circle = point.buffer(buffer_radius)

#     # Check if this new circle overlaps any existing plot
#     if circle.intersects(sampled_union):
#         attempts += 1
#         continue

#     new_plots.append(circle)
#     # Add to union to prevent overlap with future points
#     sampled_union = sampled_union.union(circle)
#     attempts += 1

# # 4. Create GeoDataFrame of new plots
# new_plots_gdf = gpd.GeoDataFrame(geometry=new_plots, crs='EPSG:32616')
# new_plots_gdf = new_plots_gdf.to_crs('EPSG:4326')

# # 5. Add to map
# Map.add_gdf(new_plots_gdf, layer_name='75 New Non-Overlapping Plots')
# Map


In [ ]:
# Add new field with all values set to 0
non_GEDI_Plots["GEDI_Centered_Plot"] = 0

# Check first few rows
print(non_GEDI_Pots.head())


In [ ]:
# Add new field with all values set to 1
sampled_plots["GEDI_Centered_Plot"] = 1

# Check first few rows
print(sampled_plots.head())


merge and export

In [ ]:
# sampled_plots_out = pd.concat([sampled_plots, non_GEDI_Pots]).drop_duplicates(subset=gedi_buffers.columns)

# # Check first few rows
# print(sampled_plots_out.head())


In [ ]:
# export_parth = '/content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/In_Situ_Plots'

In [ ]:
# import os

# # Define export path and filename
# export_path = '/content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/In_Situ_Plots'
# os.makedirs(export_path, exist_ok=True)  # make sure folder exists

# shp_path = os.path.join(export_path, "sampled_plots_out.shp")

# # Save shapefile
# sampled_plots_out.to_file(shp_path, driver='ESRI Shapefile')

# print(f"Shapefile saved to: {shp_path}")


In [ ]:
# import os

# # Define export path and filename
# export_path = '/content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/In_Situ_Plots'
# os.makedirs(export_path, exist_ok=True)  # make sure folder exists

# csv_path = os.path.join(export_path, "sampled_plots_out.csv")

# # Drop geometry for CSV export
# sampled_plots_out_no_geom = sampled_plots_out.drop(columns="geometry")

# # Save CSV
# sampled_plots_out_no_geom.to_csv(csv_path, index=False)

# print(f"CSV saved to: {csv_path}")


In [ ]:
import geopandas as gpd
from shapely.geometry import Point

# Ensure sampled_plots is a GeoDataFrame with geometry
# Convert lat/lon to Point geometry
sampled_plots_gdf = gpd.GeoDataFrame(
    sampled_plots,
    geometry=gpd.points_from_xy(sampled_plots['lon'], sampled_plots['lat']),
    crs='EPSG:4326'
)

# Project to UTM for accurate buffering
utm_crs = 'EPSG:32616'  # adjust if needed
sampled_plots_gdf = sampled_plots_gdf.to_crs(utm_crs)

# Create 25m diameter (12.5m radius) circular polygons
sampled_plots_gdf['geometry'] = sampled_plots_gdf.geometry.buffer(12.5)

# Back to WGS84 for consistency
sampled_plots_gdf = sampled_plots_gdf.to_crs('EPSG:4326')

# Now both datasets have polygon geometry in the same CRS
combined_gdf = pd.concat([non_GEDI_Plots, sampled_plots_gdf], ignore_index=True)

# Ensure final is a GeoDataFrame
combined_gdf = gpd.GeoDataFrame(combined_gdf, geometry='geometry', crs='EPSG:4326')

# Export shapefile
export_path = '/content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/In_Situ_Plots'
combined_gdf.to_file(os.path.join(export_path, 'all_plots.shp'))

# Export CSV (keep WKT geometry so you still have polygon coordinates)
combined_gdf.to_csv(os.path.join(export_path, 'all_plots.csv'), index=False)


check to ensure export looks right

In [ ]:
import geopandas as gpd
import pandas as pd
from shapely import wkt
import matplotlib.pyplot as plt

# Path to your CSV
csv_path = "/content/drive/MyDrive/PhD_Main_Folder/Geo_Data/Sewanee_DL_Project/In_Situ_Plots/all_plots.csv"

# Read CSV
df = pd.read_csv(csv_path)

# Convert 'geometry' column from WKT to shapely geometry
df['geometry'] = df['geometry'].apply(wkt.loads)

# Convert to GeoDataFrame
gdf = gpd.GeoDataFrame(df, geometry='geometry', crs='EPSG:4326')

# Plot
fig, ax = plt.subplots(figsize=(8, 8))
gdf.plot(ax=ax, edgecolor='black', facecolor='lightblue')
ax.set_title("All Plots", fontsize=14)
plt.show()


statstically valid?

In [ ]:
import geopandas as gpd
from esda import Moran
from libpysal.weights import KNN

gdf = gpd.read_file("plots.shp")

# Area-based weights for later analysis
area_per_unit = gdf.groupby("unit")["unit_area"].first()
weights = area_per_unit / gdf["unit"].value_counts()

# Moran's I
coords = list(zip(gdf.geometry.x, gdf.geometry.y))
w = KNN.from_array(coords, k=4)
mi = Moran(gdf["response_var"], w)
print(mi.I, mi.p_norm)
